In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('')))
from config import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
from collections import Counter
from torchvision import transforms
from PIL import Image
from pathlib import Path
from src.dataset import get_dataloaders, get_class_names

config = Config()
class_names = get_class_names(config)
print("Classes:", class_names)

In [ ]:
# Class distribution bar chart
dataset_root = Path(config.DATASET_ROOT)
class_counts = {}
for cls_name in class_names:
    cls_dir = dataset_root / cls_name
    if cls_dir.exists():
        class_counts[cls_name] = len(list(cls_dir.glob("*.*")))

plt.figure(figsize=(12, 6))
plt.bar(class_counts.keys(), class_counts.values(), color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Number of Images')
plt.title('Class Distribution in Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Sample grid — 3 images per class
num_classes = len(class_names)
samples_per_class = 3
fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(10, 3 * num_classes))

if num_classes > 0:
    for i, cls_name in enumerate(class_names):
        cls_dir = dataset_root / cls_name
        if cls_dir.exists():
            images = list(cls_dir.glob("*.*"))[:samples_per_class]
            for j, img_path in enumerate(images):
                img = Image.open(img_path)
                ax = axes[i, j] if num_classes > 1 else axes[j]
                ax.imshow(img)
                ax.axis('off')
                if j == 1:
                    ax.set_title(cls_name)
    plt.tight_layout()
    plt.show()

In [ ]:
# DataLoader sanity check
try:
    train_loader, val_loader, test_loader = get_dataloaders(config)
    images, labels = next(iter(train_loader))
    print(f"Train Batch Image Shape: {images.shape}")
    print(f"Train Batch Label Shape: {labels.shape}")

    # Show augmented vs non-augmented for the same image
    # Pick a random image from the dataset directly
    sample_img_path = list(dataset_root.glob("*/*.*"))[0]
    raw_image = Image.open(sample_img_path).convert("RGB")
    
    # Use the transforms from dataset
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]
    
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(config.IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor()
    ])
    
    aug_image_tensor = train_transform(raw_image)
    aug_image_np = aug_image_tensor.permute(1, 2, 0).numpy()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(raw_image)
    ax1.set_title("Original Image")
    ax1.axis('off')
    
    ax2.imshow(aug_image_np)
    ax2.set_title("Augmented Image (Before Normalize)")
    ax2.axis('off')
    plt.show()

except Exception as e:
    print(f"Could not load dataloaders (might be empty dataset): {e}")

In [ ]:
# Class imbalance check
if class_counts:
    counts = list(class_counts.values())
    print(f"Minimum samples per class: {np.min(counts)}")
    print(f"Maximum samples per class: {np.max(counts)}")
    print(f"Mean samples per class: {np.mean(counts):.2f}")
else:
    print("No classes found to compute imbalance.")